<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex02-pytorch-and-autograd/Ex02_03_training_loop_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_2 · Notebook 03 — the training loop

Forward, loss, `zero_grad`, `backward`, `step`. Five lines, fitting a curve.

## Where this sits

Notebook 02 did the hard half: differentiating with respect to an **input**,
which is the mechanism of Part 2 and the thing almost nobody arrives knowing.
This notebook does the familiar half — differentiating with respect to the
**weights**, and using the result to improve them.

It is deliberately in this order. Presented the other way round, the training
loop makes autograd look like a component of neural-network training rather
than a general tool that training happens to use. You will now recognise
`loss.backward()` as the same operation as `grad(u, x)`, aimed at a different
tensor.

## The problem

An impact test on a lightly damped structure. You strike it, record the
acceleration, and get a decaying oscillation:

$$y(t) = e^{-t/2}\,\sin(2\pi t)$$

sampled at sixty irregular times over two seconds, with measurement noise of
standard deviation 0.03. You will fit a small network to it.

Be clear about what that does and does not achieve. Fitting a network to this
curve is **not** the right way to solve it — you know the closed form, and if
you did not, `scipy.optimize.curve_fit` with three parameters would beat the
network on every measure that matters: accuracy, speed, interpretability, and
the fact that the fitted parameters would be the damping ratio and the natural
frequency rather than two thousand meaningless numbers.

The network is here because the loop is the object of study. Section 7 makes
the point sharply by asking the trained network to extrapolate.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_2_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex02-pytorch-and-autograd/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup -------------------------------------------------------------
# Needs Ex_2_core.py alongside this notebook.
import os
for f in ("Ex_2_core.py",):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from Ex_2_core import *                             # noqa: F401,F403
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

set_seed(88)
print("setup complete | device:", DEVICE)

## 1 · The five lines

Every training loop in this course, and in Part 2, is this:

```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(n_epochs):
    pred = model(t)                        # 1. forward
    loss = ((pred - y) ** 2).mean()        # 2. loss
    optimizer.zero_grad()                  # 3. clear old gradients
    loss.backward()                        # 4. gradients w.r.t. every parameter
    optimizer.step()                       # 5. one step downhill
```

Take them one at a time, because each has something in it worth knowing.

**1. Forward.** `model(t)` runs the network and builds the graph. Nothing is
optimised; this is just evaluation, with a record kept.

**2. Loss.** A single scalar measuring how wrong the prediction is. It must be
a scalar — shape `()` — because `backward()` computes a gradient, and a
gradient of a vector with respect to parameters is a Jacobian. `.mean()` is
what turns sixty squared errors into one number.

Note also that the loss is where all the engineering lives. Here it is the mean
squared error against data. In Part 2 it is the mean squared **residual** of a
differential equation, with no data anywhere in it, and every other line of this
loop stays exactly as written.

**3. `zero_grad()`.** PyTorch **accumulates** gradients: `backward()` adds to
`.grad` rather than replacing it. Forget this call and step *k* uses the sum of
the gradients from steps 1 to *k*, which is a direction that gets longer and
staler as training goes on. Section 5 makes you do it on purpose.

Why would a library accumulate by default? Because it lets you split a batch
too large for memory into pieces, call `backward()` on each, and step once with
the total — and because a recurrent network needs to accumulate contributions
from every time step. Both are real needs. The cost is one line in every loop
you will ever write.

**4. `backward()`.** The backward pass. It walks the graph from the loss to
every leaf with `requires_grad=True` — which is every parameter of the model —
and deposits the gradient in each parameter's `.grad`.

**5. `step()`.** The optimiser updates each parameter using its `.grad`. For
Adam that means a per-parameter step size derived from running averages of the
gradient and its square. What matters here is that the optimiser knows nothing
about your model: it was handed `model.parameters()` at construction and it
reads `.grad` off them. That separation is why you can swap Adam for L-BFGS in
Part 2 by changing one line.

**The order matters.** `zero_grad` must come before `backward`, and `step` after
it. Any other arrangement either steps on stale gradients or throws away the
ones just computed.

## 2 · The data

`vibration_data` returns `t` and `y` as float32 columns of shape `(60, 1)` —
the shape convention from notebook 01, so that `nn.Linear` sees a batch of
sixty samples with one feature each.

Look at the figure before fitting anything. Two oscillations, decaying, with
the noise visible but not dominant. A straight line cannot fit this and neither
can a quadratic, so if the trained network reproduces it, something was
genuinely learned.

In [ ]:
t, y = vibration_data(n=60, noise=0.03, seed=88)
print("t:", tuple(t.shape), t.dtype, " y:", tuple(y.shape), y.dtype)
print("t requires_grad:", t.requires_grad,
      " <- not needed here: we differentiate w.r.t. the WEIGHTS, not the input")

t_dense = torch.linspace(0.0, 2.0, 400).reshape(-1, 1)
truth = vibration_truth(to_numpy(t_dense).ravel())

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(to_numpy(t_dense).ravel(), truth, "-", linewidth=1.5, label="true response")
ax.plot(to_numpy(t).ravel(), to_numpy(y).ravel(), "o", markersize=4, label="measured")
engineering_axes(ax, "time t [s]", "acceleration y [m/s$^2$]",
                 title="impact test: free decay of a damped structure", legend=True)
plt.show()

**What you should see.** A decaying sinusoid with sixty markers scattered close
to it, denser in some places than others because the sampling times are random.

Note the line about `requires_grad`. In notebook 02 the input carried
`requires_grad=True` because the derivative *with respect to it* was the point.
Here it does not, because the gradients we want are with respect to the
weights, and those are tracked automatically. Same engine, different target.

## 3 · Write the loop

Write `train(model, t, y, epochs=3000, lr=5e-3)`. It must:

1. build an Adam optimiser over `model.parameters()` with learning rate `lr` —
   **outside** the loop, because Adam keeps running averages per parameter and
   rebuilding it every epoch would throw them away and cripple it;
2. loop `epochs` times, doing the five steps in the right order;
3. append the loss **as a float** to a list, using `loss.item()`;
4. return that list.

Point 3 is not a style preference. Appending the loss *tensor* keeps the whole
graph for that epoch alive, because the tensor holds a reference to it. Do that
for three thousand epochs and you have three thousand graphs in memory. The
symptom is a notebook that slows down and then dies; the fix is `.item()`, and
the same applies to anything else you log during training.

Use the mean squared error written out — `((pred - y) ** 2).mean()` — rather
than `nn.MSELoss()`. There is nothing wrong with the library version, but in
Part 2 you will be writing losses that no library provides, and the habit of
writing the expression is worth more than the four characters it saves.

About ten lines.

In [ ]:
# TODO 1 --- the training loop ---------------------------------------------------------
# Five `...` to replace, in this order:
#   line 1  ->  torch.optim.Adam(model.parameters(), lr=lr)     built ONCE, before the loop
#   line 2  ->  model(t)                                        forward pass
#   line 3  ->  ((pred - y) ** 2).mean()                        mean squared error
#   line 4  ->  optimizer.zero_grad()                           clear last epoch's gradients
#   line 5  ->  loss.item()                                     a float, not the tensor
def train(model, t, y, epochs=3000, lr=5e-3):
    """Fit `model` to (t, y) by minimising the mean squared error.

    Returns the list of loss values, one per epoch.
    """
    optimizer = ...                               # <- torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        pred = ...                                # <- model(t)
        loss = ...                                # <- ((pred - y) ** 2).mean()
        ...                                       # <- optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(...)                       # <- loss.item()
    return history
# ------------------------------------------------------------------------------

In [ ]:
set_seed(88)
model = mlp(n_in=1, n_out=1, n_hidden=32, n_layers=3, activation="tanh")
print(count_parameters(model), "parameters for 60 data points -",
      "far more parameters than data, which is normal and is section 7's problem")

history = train(model, t, y, epochs=3000, lr=5e-3)

print()
print(f"  first loss {history[0]:.4e}")
print(f"  last loss  {history[-1]:.4e}")
print(f"  fell by a factor of {history[0] / history[-1]:.0f}")

**What you should see.** A first loss of order 0.3 and a final loss of order
1e-3, a fall of two to three orders of magnitude. The exact numbers move with
the torch version; the order of magnitude should not.

If the loss did not fall at all, check that `zero_grad` is inside the loop and
that `step` comes after `backward`. If it became `nan`, the learning rate is
too large — halve it.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
plot_loss(history, ax=ax, label="training loss", title="Adam, lr = 5e-3")
plt.show()

**What you should see.** A curve on a logarithmic y-axis, falling steeply for
the first few hundred epochs, then flattening as it approaches the noise floor
of the data.

It flattens for a good reason. The data carry noise of standard deviation 0.03,
so a mean squared error of 0.03² = 9e-4 is the best any model can do without
fitting the noise itself. A loss that settles near 1e-3 has extracted
everything there is. A loss that keeps falling well below it is memorising
noise, which is overfitting, and is the subject of Ex_6.

**Always on a log axis.** Plot this linearly and everything after epoch 300 is
pressed flat against zero.

## 4 · Evaluate

Training loss tells you how well the model reproduces the points it was shown.
It says nothing about the points it was not. Evaluate on a dense grid and
compare with the true curve.

Compute:

| name | requirement |
|---|---|
| `pred_dense` | the model's prediction at `t_dense`, shape `(400, 1)`, as a **NumPy array of shape `(400,)`** after conversion |
| `rmse_truth` | the RMSE between `pred_dense` and `truth`, a float |
| `rmse_data` | the RMSE between the model's prediction at `t` and the measured `y`, a float |

Two points of technique.

**Wrap the prediction in `torch.no_grad()`.** You are not differentiating, so
there is no reason to build a graph. It is faster and it makes the intent
plain. In a larger model it is also the difference between fitting in memory
and not.

**Convert with `to_numpy`**, which detaches, moves to the CPU, and converts, in
that order. Then `.ravel()` to get a flat `(400,)` array to compare against
`truth`, which NumPy produced and which is already flat. This is exactly the
`(N,)` against `(N, 1)` situation from Ex_1: if you subtract without
flattening, you will get a `(400, 400)` matrix and an RMSE that is wrong and
looks plausible. Print the shapes.

In [ ]:
# TODO 2 --- evaluate on the dense grid -------------------------------------------------
# Three `...` to replace, one per line:
#   pred_dense  ->  to_numpy(model(t_dense)).ravel()                        shape (400,)
#   rmse_truth  ->  float(np.sqrt(((pred_dense - truth) ** 2).mean()))
#   rmse_data   ->  float(((model(t) - y) ** 2).mean().sqrt())
with torch.no_grad():
    pred_dense = ...                              # <- to_numpy(model(t_dense)).ravel()
    rmse_truth = ...                              # <- float(np.sqrt(((pred_dense - truth) ** 2).mean()))
    rmse_data  = ...                              # <- float(((model(t) - y) ** 2).mean().sqrt())
print("pred_dense:", pred_dense.shape, " truth:", truth.shape)
assert not any(v is ... for v in (pred_dense, rmse_truth, rmse_data)), "TODO 2: replace the three ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("pred_dense", pred_dense, (400,))
print(f"  RMSE against the true curve : {rmse_truth:.4f}")
print(f"  RMSE against the noisy data : {rmse_data:.4f}")
print(f"  the noise in the data was     0.0300")

fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.plot(to_numpy(t_dense).ravel(), truth, "-", linewidth=1.5, label="true response")
ax.plot(to_numpy(t_dense).ravel(), pred_dense, "--", linewidth=2.0, label="network")
ax.plot(to_numpy(t).ravel(), to_numpy(y).ravel(), "o", markersize=4, label="measured")
engineering_axes(ax, "time t [s]", "acceleration y [m/s$^2$]",
                 title="fit on the training interval", legend=True)
plt.show()

**What you should see.** Both RMSE values around 0.02 to 0.04, and a dashed
network curve lying close to the solid true one across the whole interval.

The interesting comparison is between the two numbers. The RMSE against the
noisy data cannot go below the noise level, because the noise is not
predictable. The RMSE against the *true* curve can, and if it is meaningfully
smaller than 0.03 then the network has done what a good fit does: averaged the
noise out rather than chased it.

If your network's curve overshoots wildly between data points in the sparse
region, it has too much capacity for the data — again, Ex_6.

## 5 · What `zero_grad` actually prevents

The best way to understand a line is to delete it.

Write `train_broken`, identical to your `train` but **with the `zero_grad` call
removed**. Everything else stays: same optimiser, same loss, same order.

Predict what will happen before you run it. Gradients accumulate, so at epoch
*k* the `.grad` of each parameter holds the sum of the gradients from all *k*
epochs so far. Early on, when successive gradients point in roughly the same
direction, that sum is simply a much larger step in that direction — so the
first few epochs may look fast. Then the accumulated vector becomes dominated
by stale gradients computed at parameter values the model has long since left,
and the steps stop bearing any relation to the current loss surface.

Use a shorter run — 1000 epochs — because the point is visible early and there
is no reason to wait.

In [ ]:
# TODO 3 --- the same loop with zero_grad removed ----------------------------------------
# One `...` to replace:  loss.backward()
# Read the loop first: it is train() with the optimizer.zero_grad() line deleted.
# That deletion is the whole experiment.
def train_broken(model, t, y, epochs=1000, lr=5e-3):
    """The same loop with the zero_grad call removed. Do not copy this into anything."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        pred = model(t)
        loss = ((pred - y) ** 2).mean()
        ...                                       # <- loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history
# ------------------------------------------------------------------------------

In [ ]:
set_seed(88)
model_ok = mlp(n_in=1, n_out=1, n_hidden=32, n_layers=3)
set_seed(88)
model_bad = mlp(n_in=1, n_out=1, n_hidden=32, n_layers=3)

hist_ok = train(model_ok, t, y, epochs=1000, lr=5e-3)
hist_bad = train_broken(model_bad, t, y, epochs=1000, lr=5e-3)

print(f"  with zero_grad    : final loss {hist_ok[-1]:.4e}")
print(f"  without zero_grad : final loss {hist_bad[-1]:.4e}")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
plot_loss(hist_ok, ax=ax, label="with zero_grad")
plot_loss(hist_bad, ax=ax, label="without zero_grad")
ax.set_title("the one line people forget")
plt.show()

**What you should see.** Two curves that start together and separate. The
correct one falls smoothly. The broken one is visibly rougher — a jagged,
oscillating curve rather than a clean descent — and ends at a higher loss.

**How much higher depends on the optimiser, and that is worth knowing.** Adam
divides each step by a running estimate of the gradient's magnitude, so
multiplying every gradient by a growing factor is partly cancelled out; the
damage that survives is *staleness*, the fact that the accumulated vector is
dominated by gradients computed at parameter values the model left long ago.
With plain SGD the same bug produces steps that grow without limit and the run
diverges to `nan` within a few hundred epochs. So the symptom ranges from
"slightly worse" to "completely destroyed" depending on what else is in the
loop — which is exactly what makes it hard to spot.

What it never does is raise an exception. There is no error message for this. A
run with a missing `zero_grad` looks like a run that trained badly, and you
would spend the afternoon changing the learning rate and the architecture
instead. That is why it is the first thing to check when a loss will not fall,
and why it is on the checklist in `Ex_2_core.py`.

## 6 · The learning rate

One hyperparameter matters more than all the others put together, and this is
it. The cell below runs the same model and the same data at three learning
rates, and you should be able to recognise all three shapes on sight for the
rest of the course.

- **Too small** (1e-4): the loss falls, smoothly, and does not get anywhere in
  the budget you gave it. The curve is still descending when training stops.
  The fix is more epochs or a larger rate, and the diagnostic is that the curve
  has no flat part at the end.
- **About right** (5e-3): a steep fall, then a flattening as the model
  approaches the noise floor.
- **Too large** (5e-1): the loss jumps around, may rise, and may go to `nan`.
  The optimiser is stepping past the minimum and out the other side.

This is a sweep of three values and it is the honest way to choose: try a few
orders of magnitude, look at the curves, and take the one that has a steep part
and a flat part. Tuning a learning rate to two significant figures is almost
always wasted effort.

In [ ]:
results = {}
for lr in (1e-4, 5e-3, 5e-1):
    set_seed(88)
    m = mlp(n_in=1, n_out=1, n_hidden=32, n_layers=3)
    results[lr] = train(m, t, y, epochs=1500, lr=lr)
    print(f"  lr = {lr:<8} final loss {results[lr][-1]:.4e}")

fig, ax = plt.subplots(figsize=(6.5, 4.0))
for lr, hist in results.items():
    plot_loss(hist, ax=ax, label=f"lr = {lr}")
ax.set_title("three learning rates, same model, same data")
plt.show()

**What you should see.** Three curves. The 1e-4 run still falling at the right
edge of the plot; the 5e-3 run down two or three decades and flattening; the
5e-1 run erratic and high, possibly with gaps in the line where the loss went
to `nan` and the log axis could not draw it.

A `nan` is worth recognising: once one appears it propagates through every
parameter on the next step, and the model is dead. Restart from a fresh model
with a smaller rate; there is no recovering the run.

## 7 · Ask it something it was not shown

The model fits the data. Now ask it about times it never saw.

The training data covered 0 to 2 seconds. The cell below evaluates the network
from −0.5 to 3.5 and plots it against the true response, which of course
continues to decay and oscillate.

In [ ]:
t_wide = torch.linspace(-0.5, 3.5, 600).reshape(-1, 1)
with torch.no_grad():
    pred_wide = to_numpy(model(t_wide)).ravel()
truth_wide = vibration_truth(to_numpy(t_wide).ravel())

fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.axvspan(0.0, 2.0, color="0.9", label="training interval")
ax.plot(to_numpy(t_wide).ravel(), truth_wide, "-", linewidth=1.5, label="true response")
ax.plot(to_numpy(t_wide).ravel(), pred_wide, "--", linewidth=2.0, label="network")
ax.plot(to_numpy(t).ravel(), to_numpy(y).ravel(), "o", markersize=4, label="measured")
engineering_axes(ax, "time t [s]", "acceleration y [m/s$^2$]",
                 title="outside the training interval", legend=True)
plt.show()

inside = np.abs(to_numpy(t_wide).ravel() - 1.0) <= 1.0
print(f"  RMSE inside the training interval  : "
      f"{np.sqrt(((pred_wide - truth_wide)[inside] ** 2).mean()):.4f}")
print(f"  RMSE outside it                    : "
      f"{np.sqrt(((pred_wide - truth_wide)[~inside] ** 2).mean()):.4f}")

**What you should see.** Inside the shaded band, the two curves agree. Outside
it, they part company immediately and completely — the network typically
flattens out or runs off in whatever direction its last piece was heading,
while the true response carries on oscillating. The RMSE outside is an order of
magnitude worse than inside, or more.

Three things to take from this, and they are not the same thing.

**It is not a bug and it is not overfitting.** The network was asked to
minimise an error on sixty points between 0 and 2. It did. Nothing in the loss
mentioned t = 3, so nothing constrained the model there, and no amount of
training or regularisation would have helped. A fitted model is a statement
about the region the data covered and about nothing else.

**This is what L4.1 slide 23 explains.** A network is built from a finite
number of pieces; outside the data it continues along whichever piece it ended
on. For a ReLU network that is literally a straight line. It is worth looking at
the shape of your curve out at t = 3.5 with that description in mind.

**And this is the argument for Part 2.** The reason a physics-informed network
behaves better outside its data is not that it has more data — it has none. It
is that its loss contains a differential equation which holds *everywhere in
the domain*, including where you never sampled. Constraining a model with
physics rather than with points is the difference between a fit and a solution,
and it is why the next ten weeks exist.

## 8 · One loop, two courses

Compare what you wrote in this notebook with what you wrote in notebook 02.

**Here — Ex_3 to Ex_6, supervised learning:**

```python
pred = model(t)
loss = ((pred - y) ** 2).mean()          # needs data
optimizer.zero_grad(); loss.backward(); optimizer.step()
```

**There — Ex_7 to Ex_12, physics-informed learning:**

```python
u = model(x)
u_xx = grad(grad(u, x), x)               # needs autograd w.r.t. the input
loss = ((u_xx - f(x)) ** 2).mean()       # needs no data at all
optimizer.zero_grad(); loss.backward(); optimizer.step()
```

The third line is identical. The optimiser, the backward pass, the accumulation
rule, the learning-rate behaviour — all unchanged. What changed is the second
line, and that is where every week of Part 2 does its engineering.

## What you have done

You can write the five-line training loop from memory and say what each line
is for; you know why the optimiser is built outside the loop and why the loss
is logged with `.item()`; you have seen what a missing `zero_grad` does and
that it raises nothing; you can recognise a learning rate that is too small,
about right, and too large from the shape of the loss curve; you know why a
loss that settles at the noise floor has done its job; and you have watched a
well-fitted model fail completely half a second outside its data.

## Ex_2 is complete

Four notebooks: the environment, tensors, autograd, and the loop.

The one that matters is 02. If you can differentiate a function with respect to
its input, take the second derivative, write a residual, and say why ReLU
cannot be used for it, you are ready for everything that follows — and the rest
of this course is those four skills applied to heat, flow, cells, robots and
grids.